# 🛑 L9 — Human-in-the-Loop : demander avant d'agir

> ⏱️ **Durée indicative : 30 à 45 minutes**  
> 🎯 **Objectif :** suspendre un agent avant un appel d'outil, inspecter sa proposition, puis la rejeter ou l'approuver explicitement.

Nous utiliserons Chinook en lecture seule. Même une requête sans danger sera interrompue pour rendre tout le cycle visible : **proposition → interruption → décision humaine → reprise**.

## 🧠 ELI5 — Le bouton « Confirmer la commande »

Imaginez un stagiaire qui prépare une commande mais ne peut pas cliquer sur **Envoyer** :

1. Mistral propose une action et ses arguments ;
2. le middleware place un cadenas juste avant l'outil ;
3. LangGraph sauvegarde le dossier et rend la main à l'humain ;
4. l'humain rejette ou approuve ;
5. le même dossier reprend exactement au point d'arrêt.

La documentation officielle [Human-in-the-Loop de LangChain](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) décrit ce cycle et ses décisions `approve`, `edit` et `reject`.

### 🗺️ Schéma mental

```text
Utilisateur → Mistral propose execute_sql(...) → 🛑 interruption
                                              ├─ reject  → outil NON exécuté
                                              └─ approve → outil exécuté → réponse
```

## 🎯 Parcours de l'atelier

- préparer Mistral et une base réellement en lecture seule ;
- construire un outil dont les exécutions sont observables ;
- configurer le middleware HITL et un checkpointer ;
- rejeter une première proposition ;
- approuver la même famille de proposition dans un autre fil ;
- terminer par un exercice sur le nombre de clients.

## 🛠️ 0. Configurer Mistral depuis le processus

Le notebook utilise seulement `MISTRAL_API_KEY` et `MISTRAL_SERVER_URL` déjà définies dans le processus. Il ne lit, n'affiche et ne modifie aucun fichier `.env`.

Nous instancions [`ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai) avec `mistral-medium-latest` et `temperature=0`.

In [21]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
import os

from langchain_mistralai import ChatMistralAI

# 📚 https://docs.langchain.com/oss/python/integrations/chat/mistralai
MODEL = "mistral-medium-latest"
mistral_model = ChatMistralAI(
    model=MODEL,
    temperature=0,
    endpoint=os.environ["MISTRAL_SERVER_URL"] + "/v1",
)

print(f"✅ Modèle configuré : {MODEL} (temperature=0)")

✅ Modèle configuré : mistral-medium-latest (temperature=0)


### 👀 Résultat attendu

Seuls le modèle et la température sont affichés. Une clé, même partiellement masquée, ne doit jamais apparaître dans les outputs.

## 🛠️ 1. Préparer Chinook en lecture seule

Le HITL apporte une validation humaine, mais il ne remplace pas les protections techniques. Nous ouvrons donc SQLite avec `mode=ro` et nous limitons l'outil à une seule requête `SELECT` ou `WITH`. 🔒

Ce code charge la base SQLite `data/Chinook.db` via un chemin absolu, vérifie que le fichier existe, puis crée une URI de connexion en lecture seule avec `mode=ro`.

La variable `readonly_uri` contient cette chaîne de connexion :

`sqlite:///file:<chemin_base>?mode=ro&uri=true`

Elle indique à SQLite d’ouvrir la base en mode lecture seule, ce qui empêche les opérations d’écriture comme `INSERT`, `UPDATE`, `DELETE` ou `DROP`. Le code se connecte ensuite à la base, récupère le schéma des tables `Employee` et `Customer`, puis confirme que le schéma a été chargé.

In [22]:
from pathlib import Path

from util.sql_db import SQLDatabase

db_path = Path("data/Chinook.db").resolve()
if not db_path.is_file():
    raise FileNotFoundError(f"Base Chinook introuvable : {db_path}")

# Défense technique indépendante du prompt et de la décision humaine.
readonly_uri = f"sqlite:///file:{db_path.as_posix()}?mode=ro&uri=true"
db = SQLDatabase.from_uri(readonly_uri)
USEFUL_TABLES = ["Employee", "Customer"]
schema_context = db.get_table_info(USEFUL_TABLES)
print("✅ Base en lecture seule ; schéma chargé :", ", ".join(USEFUL_TABLES))

✅ Base en lecture seule ; schéma chargé : Employee, Customer


In [23]:
print(schema_context)


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Customer table:
CustomerId	FirstName	LastName	Company	Address	City	State	Country	PostalCode	Phone	Fax	Email	SupportRepId
1	Luís	Gonçalves	Embraer - Empresa Brasileira de Aeronáutica S.A.	Av. Brigadeiro Faria Lima, 2170	São José dos Campos	SP	Brazil	12227-000	+55 (12) 3923-5555	+55 (12) 3923-5566	luisg@embraer.com.br	3
2	Leonie	Köhler	None	Theodor-Heuss-Straße 34	Stuttgart	None	Germany	70174	+49 0711 2842222	None	leonekohler@surfeu.de	5
3	François	Tremblay	None	1498 rue Bélanger	Montréal	QC

### 🔍 Pourquoi donner le schéma ?

Sans carte, un modèle peut inventer une table `employees`. Chinook utilise les noms exacts `Employee` et `Customer`. Le schéma sera inclus dans le prompt et la requête de démonstration sera explicitement recommandée.

In [24]:
EMPLOYEE_ORACLE_SQL = "SELECT FirstName, LastName FROM Employee ORDER BY EmployeeId"
CUSTOMER_COUNT_ORACLE_SQL = "SELECT COUNT(*) AS NombreClients FROM Customer"

employee_oracle = db.run(EMPLOYEE_ORACLE_SQL)
customer_count_oracle = db.run(CUSTOMER_COUNT_ORACLE_SQL)
print("Employés attendus :", employee_oracle)
print("Nombre de clients attendu :", customer_count_oracle)

Employés attendus : [('Andrew', 'Adams'), ('Nancy', 'Edwards'), ('Jane', 'Peacock'), ('Margaret', 'Park'), ('Steve', 'Johnson'), ('Michael', 'Mitchell'), ('Robert', 'King'), ('Laura', 'Callahan')]
Nombre de clients attendu : [(59,)]


### 👀 Résultat attendu

- huit employés, de **Andrew Adams** à **Laura Callahan** ;
- `59` clients.

Ces valeurs sont des oracles déterministes. La prose de Mistral peut changer, mais pas les données retournées par SQLite.

## 🛠️ 2. Créer l'outil SQL observable

Le décorateur [`@tool`](https://docs.langchain.com/oss/python/langchain/tools) expose la fonction à l'agent. Le [`runtime context`](https://docs.langchain.com/oss/python/langchain/runtime) injecte la base sans laisser Mistral choisir la connexion.

Un journal Python `EXECUTED_QUERIES` nous donnera une preuve simple : une proposition interrompue ou rejetée ne doit pas y apparaître.

In [25]:
from dataclasses import dataclass

from langchain_core.tools import tool
from langgraph.runtime import get_runtime


@dataclass
class RuntimeContext:
    db: SQLDatabase


FORBIDDEN_SQL = {
    "insert", "update", "delete", "alter", "drop", "create",
    "replace", "truncate", "attach", "detach", "pragma",
}
EXECUTED_QUERIES: list[str] = []

# on définit une fonction de validation pour s'assurer que la requête SQL 
# est en lecture seule et ne contient pas d'opérations interdites.
def validate_readonly_query(query: str) -> str:
    """Valide une unique requête SELECT/CTE."""
    # On supprime les espaces superflus et le point-virgule final.
    statement = query.strip()
    body = statement[:-1].strip() if statement.endswith(";") else statement
    lowered = body.lower()
    # On vérifie que la requête commence par SELECT ou WITH, et qu'elle ne contient pas de point-virgule.
    if not lowered.startswith(("select ", "with ")):
        raise ValueError("Seules les requêtes SELECT ou WITH sont autorisées.")
    if ";" in body:
        raise ValueError("Une seule instruction SQL est autorisée.")
    words = set(lowered.replace("(", " ").replace(")", " ").split())
    # On vérifie que la requête ne contient pas de mots interdits.
    if words.intersection(FORBIDDEN_SQL):
        raise ValueError("La requête contient une opération interdite.")
    return body


# 📚 Tools : https://docs.langchain.com/oss/python/langchain/tools
# 📚 Runtime : https://docs.langchain.com/oss/python/langchain/runtime
@tool
def execute_sql(query: str) -> str:
    """Exécute une seule requête SQLite en lecture seule sur Chinook."""
    runtime = get_runtime(RuntimeContext)
    safe_query = validate_readonly_query(query)
    EXECUTED_QUERIES.append(safe_query)
    try:
        return runtime.context.db.run(safe_query)
    except Exception as error:
        return f"Erreur SQL : {error}"

### 🔍 Trois couches différentes

| Couche | Rôle |
|---|---|
| Prompt | guide Mistral vers une requête pertinente |
| HITL | exige une décision avant l'appel d'outil |
| Lecture seule + validation Python | bloque techniquement les écritures |

⚠️ Une approbation humaine n'assainit pas magiquement une requête. Il faut lire le nom de l'action **et ses arguments** avant de décider.

## 🛠️ 3. Installer le point d'arrêt HITL

`HumanInTheLoopMiddleware` interrompt `execute_sql`. [`InMemorySaver`](https://docs.langchain.com/oss/python/langchain/short-term-memory) sauvegarde l'état nécessaire à la reprise. Le [guide HITL officiel](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) précise qu'il faut reprendre avec le **même** `thread_id`.

In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM_PROMPT = f"""Tu es un analyste SQLite prudent.

Règles :
- Utilise obligatoirement `execute_sql` pour répondre depuis Chinook.
- Produis une seule requête SELECT ou WITH.
- Utilise exactement les noms de tables et colonnes du schéma.
- Pour lister les employés, utilise :
  SELECT FirstName, LastName FROM Employee ORDER BY EmployeeId
- Pour compter les clients, utilise :
  SELECT COUNT(*) AS NombreClients FROM Customer
- Après un rejet, explique que l'action a été refusée sans proposer un nouvel outil.
- Après un résultat réussi, réponds en français sans autre appel d'outil.

Schéma :
{schema_context}
"""

# 📚 https://docs.langchain.com/oss/python/langchain/human-in-the-loop
# 📚 https://docs.langchain.com/oss/python/langchain/short-term-memory

## `InMemorySaver`

`InMemorySaver` est un **checkpointer** LangGraph/LangChain : il sert à sauvegarder l’état d’un agent pendant son exécution.

```python
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
```
Il stocke les informations en mémoire RAM, par exemple l’historique court d’une conversation, l’état d’un thread via `thread_id`, les interruptions `human-in-the-loop`, ou encore les décisions humaines comme `approve`, `edit`, `reject` et `respond`.
Dans un workflow human-in-the-loop, il permet à l’agent de se mettre en pause avant une action sensible, puis de reprendre après validation humaine.

`InMemorySaver()` signifie : “crée un espace de sauvegarde temporaire en mémoire pour l’état de l’agent”.

⚠️ Comme le stockage est en mémoire, les données sont perdues si le processus Python s’arrête. Pour la production, il faut préférer un checkpointer persistant comme PostgreSQL ou MongoDB.

In [27]:
# InMemorySaver est un checkpointer simple qui stocke les décisions humaines en mémoire.
checkpointer = InMemorySaver()

# On crée un agent avec HumanInTheLoopMiddleware pour intercepter les appels à execute_sql.
agent_hitl = create_agent(
    model=mistral_model,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    # on définit le checkpointer pour stocker les décisions humaines.
    checkpointer=checkpointer,
    # on définit le middleware pour gérer les décisions humaines.
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "execute_sql": {"allowed_decisions": ["approve", "reject"]}
            }
        )
    ],
)

QUESTION_EMPLOYES = (
    "Utilise execute_sql pour lister les prénoms et noms de tous les employés, "
    "dans l'ordre de leur identifiant."
)

### 🗺️ Ce que LangGraph doit conserver

Au moment de l'interruption, l'outil n'a pas encore tourné. Le checkpointer conserve durablement **dans le contexte du processus** :

- les messages ;
- l'appel d'outil proposé ;
- la position exacte dans le graphe.

`InMemorySaver` suffit pour l'atelier, mais son contenu disparaît à l'arrêt du kernel. Une application durable utiliserait un checkpointer persistant.

## 🔍 4. Construire notre fiche de contrôle

Cette fonction refuse de décider à l'aveugle : elle vérifie la présence d'une interruption, affiche l'action et ses arguments, puis valide localement que le SQL est en lecture seule.

`assert` sert à vérifier qu’une condition attendue est vraie.  
Si la condition est fausse, Python arrête le programme avec une `AssertionError`.

Dans ce code :

```python
assert name == "execute_sql"
assert "query" in arguments
```
on vérifie que l’action suspendue est bien execute_sql et qu’elle contient une requête SQL dans `arguments["query"]`.
Ces vérifications protègent la suite du code avant d’appeler :
```python
validate_readonly_query(arguments["query"])
```


In [28]:
def inspect_pending_action(result: dict) -> dict:
    """Affiche et valide la première action suspendue."""
    if "__interrupt__" not in result:
        raise AssertionError("Aucune interruption reçue.")

    interrupt = result["__interrupt__"][0]
    payload = getattr(interrupt, "value", interrupt)
    requests = payload.get("action_requests", [])
    if not requests:
        raise AssertionError("L’interruption ne contient aucune action.")

    action = requests[0]
    name = action.get("name")
    arguments = action.get("args", {})
    description = action.get("description", "")

    print("🛑 Action proposée :", name)
    print("🧾 Arguments :", arguments)
    print("💬 Description :", description)

    assert name == "execute_sql"
    assert "query" in arguments
    validate_readonly_query(arguments["query"])
    return action

## ▶️ 5. Scénario A — Rejeter l'action

Ce scénario utilise le fil `hitl-rejet-01`. Nous arrêterons volontairement l'accès, même si la requête est en lecture seule.

### 🤔 Pause prédiction

Combien de requêtes doivent figurer dans `EXECUTED_QUERIES` juste après l'interruption ? Et après le rejet ?

Prédiction attendue : **zéro dans les deux cas**.

In [29]:
CONFIG_REJET = {"configurable": {"thread_id": "hitl-rejet-01"}}
queries_before_rejection = len(EXECUTED_QUERIES)

proposition_rejet = agent_hitl.invoke(
    {"messages": [{"role": "user", "content": QUESTION_EMPLOYES}]},
    config=CONFIG_REJET,
    context=RuntimeContext(db=db),
)

assert "__interrupt__" in proposition_rejet
assert len(EXECUTED_QUERIES) == queries_before_rejection
print("✅ Proposition suspendue ; aucune requête exécutée.")

✅ Proposition suspendue ; aucune requête exécutée.


In [30]:
action_rejet = inspect_pending_action(proposition_rejet)
print("👤 Décision humaine choisie : REJET")

🛑 Action proposée : execute_sql
🧾 Arguments : {'query': 'SELECT FirstName, LastName FROM Employee ORDER BY EmployeeId'}
💬 Description : Tool execution requires approval

Tool: execute_sql
Args: {'query': 'SELECT FirstName, LastName FROM Employee ORDER BY EmployeeId'}
👤 Décision humaine choisie : REJET


### 👀 Observation avant décision

Vous avez vu le nom `execute_sql` et le texte SQL **avant** toute exécution. Le journal est encore vide : l'interruption est bien placée devant l'outil.

In [31]:
from langgraph.types import Command

# 📚 Reprendre une interruption avec Command :
# https://docs.langchain.com/oss/python/langchain/human-in-the-loop
resultat_rejet = agent_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "Démonstration pédagogique : accès refusé.",
                }
            ]
        }
    ),
    config=CONFIG_REJET,
    context=RuntimeContext(db=db),
)

assert "__interrupt__" not in resultat_rejet
assert len(EXECUTED_QUERIES) == queries_before_rejection
print("Réponse après rejet :", resultat_rejet["messages"][-1].content)
print("✅ Requêtes réellement exécutées :", EXECUTED_QUERIES)

Réponse après rejet : L'action a été refusée.
✅ Requêtes réellement exécutées : []


### 👀 Observation après reprise

`Command` a repris **le même fil**, mais la décision `reject` a remplacé l'exécution par un retour de refus. Le journal n'a pas changé.

⚠️ Il n'y a aucune boucle automatique d'approbation : une nouvelle interruption serait une nouvelle décision à présenter à l'humain.

## ▶️ 6. Scénario B — Approuver après inspection

Nous recommençons dans un fil distinct, `hitl-approbation-01`. Ainsi, le rejet précédent ne contamine pas ce scénario.

### 🤔 Pause prédiction

Avant l'approbation, le journal doit rester inchangé. Après la reprise, il doit contenir **exactement une requête supplémentaire** et le résultat SQLite doit comporter huit employés.

In [32]:
CONFIG_APPROBATION = {
    "configurable": {"thread_id": "hitl-approbation-01"}
}
queries_before_approval = len(EXECUTED_QUERIES)

proposition_approbation = agent_hitl.invoke(
    {"messages": [{"role": "user", "content": QUESTION_EMPLOYES}]},
    config=CONFIG_APPROBATION,
    context=RuntimeContext(db=db),
)

assert "__interrupt__" in proposition_approbation
assert len(EXECUTED_QUERIES) == queries_before_approval
print("✅ Deuxième proposition suspendue ; outil toujours non exécuté.")

✅ Deuxième proposition suspendue ; outil toujours non exécuté.


In [33]:
action_approbation = inspect_pending_action(proposition_approbation)
assert "Employee" in action_approbation["args"]["query"]
print("👤 Décision humaine choisie : APPROBATION")

🛑 Action proposée : execute_sql
🧾 Arguments : {'query': 'SELECT FirstName, LastName FROM Employee ORDER BY EmployeeId'}
💬 Description : Tool execution requires approval

Tool: execute_sql
Args: {'query': 'SELECT FirstName, LastName FROM Employee ORDER BY EmployeeId'}
👤 Décision humaine choisie : APPROBATION


In [34]:
# Même CONFIG_APPROBATION : LangGraph retrouve le point d’arrêt.
resultat_approbation = agent_hitl.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=CONFIG_APPROBATION,
    context=RuntimeContext(db=db),
)

assert "__interrupt__" not in resultat_approbation
assert len(EXECUTED_QUERIES) == queries_before_approval + 1

tool_outputs = [
    str(message.content)
    for message in resultat_approbation["messages"]
    if getattr(message, "type", None) == "tool"
]
assert tool_outputs
combined_tool_output = " ".join(tool_outputs)
assert "Erreur SQL" not in combined_tool_output
assert "Andrew" in combined_tool_output and "Laura" in combined_tool_output

print("✅ Requête exécutée :", EXECUTED_QUERIES[-1])
print("📦 Résultat brut de l’outil :", tool_outputs[-1])
print("💬 Réponse finale :", resultat_approbation["messages"][-1].content)

✅ Requête exécutée : SELECT FirstName, LastName FROM Employee ORDER BY EmployeeId
📦 Résultat brut de l’outil : [('Andrew', 'Adams'), ('Nancy', 'Edwards'), ('Jane', 'Peacock'), ('Margaret', 'Park'), ('Steve', 'Johnson'), ('Michael', 'Mitchell'), ('Robert', 'King'), ('Laura', 'Callahan')]
💬 Réponse finale : Voici la liste des prénoms et noms de tous les employés, classés par identifiant :

- Andrew Adams
- Nancy Edwards
- Jane Peacock
- Margaret Park
- Steve Johnson
- Michael Mitchell
- Robert King
- Laura Callahan


### 👀 Observation après approbation

Le journal contient maintenant une requête de plus. Le `ToolMessage` prouve que SQLite a retourné les employés, puis Mistral a formulé la réponse finale.

Le modèle a **proposé** ; l'humain a **autorisé** ; Python et SQLite ont **exécuté**.

## 🔍 Comparaison des décisions

| Moment | Rejet | Approbation |
|---|---:|---:|
| Action et arguments inspectés | ✅ | ✅ |
| Même `thread_id` à la reprise | ✅ | ✅ |
| `execute_sql` réellement exécuté | ❌ | ✅ |
| Résultat SQLite transmis au modèle | ❌ | ✅ |

🧠 Le checkpointer rend la reprise possible ; la décision détermine si l'outil franchit le point d'arrêt.

## 🧪 Micro-exercice — Compter les clients

Créez un troisième fil `hitl-exercice-clients-01` :

1. demandez à l'agent de compter les clients ;
2. inspectez `execute_sql` et sa requête ;
3. vérifiez que le journal n'a pas changé ;
4. approuvez avec `Command` et le même `thread_id` ;
5. vérifiez que SQLite retourne `59`.

In [35]:
QUESTION_CLIENTS = "Utilise execute_sql pour compter exactement tous les clients."
CONFIG_EXERCICE = {
    "configurable": {"thread_id": "hitl-exercice-clients-01"}
}

# TODO 1 : mémorisez len(EXECUTED_QUERIES).
# TODO 2 : invoquez agent_hitl avec QUESTION_CLIENTS.
# TODO 3 : appelez inspect_pending_action(...).
# TODO 4 : reprenez avec Command et une décision approve.
# TODO 5 : prouvez que 59 vient du ToolMessage.

### ✅ Critères de réussite

- une interruption apparaît avant l'outil ;
- l'action inspectée est `execute_sql` ;
- la requête cible exactement `Customer` ;
- le journal ne change qu'après approbation ;
- le résultat brut contient `59`.

<details>
<summary>✅ Afficher la correction</summary>

```python
avant_exercice = len(EXECUTED_QUERIES)
proposition_clients = agent_hitl.invoke(
    {"messages": [{"role": "user", "content": QUESTION_CLIENTS}]},
    config=CONFIG_EXERCICE,
    context=RuntimeContext(db=db),
)
action_clients = inspect_pending_action(proposition_clients)
assert "Customer" in action_clients["args"]["query"]
assert len(EXECUTED_QUERIES) == avant_exercice

resultat_clients = agent_hitl.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=CONFIG_EXERCICE,
    context=RuntimeContext(db=db),
)
assert len(EXECUTED_QUERIES) == avant_exercice + 1
sorties_outil = [
    str(message.content)
    for message in resultat_clients["messages"]
    if getattr(message, "type", None) == "tool"
]
assert sorties_outil and "59" in " ".join(sorties_outil)
print(resultat_clients["messages"][-1].content)
```

</details>

## ⚠️ Limites de sécurité à retenir

- HITL n'est pas un système d'autorisation complet : l'interface humaine doit authentifier la personne qui décide.
- Un humain peut approuver une mauvaise requête ; il faut montrer les arguments sans les tronquer.
- `InMemorySaver` perd son état au redémarrage. Pour une reprise durable, utilisez un checkpointer persistant.
- Le prompt peut être contourné ; la lecture seule SQLite et la validation Python restent nécessaires.
- Ne placez jamais de secret dans la description d'une action ou dans l'état affiché.
- Dans une vraie application, journalisez la décision, l'identité du décideur et l'action approuvée.

## ✅ Acquis de la leçon

- `HumanInTheLoopMiddleware` interrompt un outil ciblé avant son exécution.
- Le payload permet d'inspecter le nom de l'action et ses arguments.
- Le checkpointer et le même `thread_id` rendent la reprise possible.
- `reject` empêche l'exécution ; `approve` la déclenche.
- `Command` transporte la décision vers le graphe suspendu.
- HITL complète les contrôles techniques, il ne les remplace pas.

## 🧭 Conclusion du parcours

Vous disposez maintenant des briques pour construire un agent outillé, mémorisé, contextualisé et supervisé. La prochaine étape naturelle est d'adapter le stockage et l'interface de validation à un contexte de production.

## 📚 Documentation officielle

- [LangChain — Human-in-the-Loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangChain — Short-term memory et checkpointers](https://docs.langchain.com/oss/python/langchain/short-term-memory)
- [LangChain — Agents et `create_agent`](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain — Tools et `@tool`](https://docs.langchain.com/oss/python/langchain/tools)
- [LangChain — Runtime context](https://docs.langchain.com/oss/python/langchain/runtime)
- [LangChain — Intégration `ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai)
- [Mistral — Function calling](https://docs.mistral.ai/studio/conversations/function-calling)